### 特徴量作成ノートブック

このノートブックでは、競艇予測のための特徴量を作成します。

#### データソース
- `data/programs.csv`: 番組表データ（1艇1行形式）
- `data/results.csv`: レース結果データ（1艇1行形式）
- `data/racers.csv`: 選手データ

In [ ]:
# データ読み込み
import category_encoders as ce
import numpy as np
import pandas as pd

In [ ]:
# programs.csvを読み込み
programs_df = pd.read_csv("data/programs.csv")

In [ ]:
print(f"programs.csvの形状: {programs_df.shape}")
print("\nカラム一覧:")
print(programs_df.columns.tolist())
print("\n先頭5行:")
programs_df.head()

programs.csvの形状: (187488, 20)

カラム一覧:
['年', '月', '日', 'レース場番号', 'レース番号', '距離', '投票締切時間', '選手登番', '年齢', '支部', '体重', '級別', '全国勝率', '全国2連率', '当地勝率', '当地2連率', 'モーター番号', 'モーター2連率', 'ボート番号', 'ボート2連率']

先頭5行:


,年,月,日,レース場番号,レース番号,距離,投票締切時間,選手登番,年齢,支部,体重,級別,全国勝率,全国2連率,当地勝率,当地2連率,モーター番号,モーター2連率,ボート番号,ボート2連率
0,2025,1,1,24,1,1800,17:41,3527,53,長崎,52,B1,5.43,32.65,5.97,40.91,57,29.20,59,36.21
1,2025,1,1,24,1,1800,17:41,4603,36,長崎,53,B1,4.98,31.40,5.32,33.33,29,39.20,53,39.32
2,2025,1,1,24,1,1800,17:41,3843,48,長崎,54,B1,4.74,28.16,5.08,29.09,43,28.35,48,36.13
3,2025,1,1,24,1,1800,17:41,5048,30,長崎,54,B1,4.32,24.19,4.60,24.19,40,24.55,51,35.29
4,2025,1,1,24,1,1800,17:41,5335,20,長崎,48,B1,2.38,6.67,2.48,10.87,65,30.58,65,37.70


In [ ]:
# データの詳細情報を確認
print("=== データ型情報 ===")
print(programs_df.dtypes)

print("\n=== 欠損値情報 ===")
print(programs_df.isnull().sum())

print("\n=== 基本統計量 ===")
print(programs_df.describe())

print("\n=== データ期間 ===")
# 年・月・日を結合して日付型に変換
programs_df["開催日"] = pd.to_datetime(
    programs_df[["年", "月", "日"]].astype(str).agg("-".join, axis=1)
)

print(f"開始日: {programs_df['開催日'].min().date()}")
print(f"終了日: {programs_df['開催日'].max().date()}")

print("\n=== レース場数 ===")
print(f"レース場数: {programs_df['レース場番号'].nunique()}")
print(f"レース場一覧: {sorted(programs_df['レース場番号'].unique())}")

print("\n=== ユニークなレース数 ===")
unique_races = (
    programs_df.groupby(["年", "月", "日", "レース場番号", "レース番号"])
    .size()
    .shape[0]
)
print(f"総レース数: {unique_races}")
print(f"総艇数: {len(programs_df)}")
print(f"1レースあたりの平均艇数: {len(programs_df) / unique_races:.1f}")

=== データ型情報 ===
年            int64
月            int64
日            int64
レース場番号       int64
レース番号        int64
距離           int64
投票締切時間      object
選手登番         int64
年齢           int64
支部          object
体重           int64
級別          object
全国勝率       float64
全国2連率      float64
当地勝率       float64
当地2連率      float64
モーター番号       int64
モーター2連率    float64
ボート番号        int64
ボート2連率     float64
dtype: object

=== 欠損値情報 ===
年          0
月          0
日          0
レース場番号     0
レース番号      0
距離         0
投票締切時間     0
選手登番       0
年齢         0
支部         0
体重         0
級別         0
全国勝率       0
全国2連率      0
当地勝率       0
当地2連率      0
モーター番号     0
モーター2連率    0
ボート番号      0
ボート2連率     0
dtype: int64

=== 基本統計量 ===
              年              月              日         レース場番号          レース番号  \
count  187488.0  187488.000000  187488.000000  187488.000000  187488.000000   
mean     2025.0       3.809908      14.859831      12.358295       6.500000   
std         0.0       1.936091       8.676709      

In [ ]:
def make_race_id(row):
    """
    年月日とレース場番号、レース番号から一意なIDを作成する関数
    """
    date_str = f"{row['年']:04d}{row['月']:02d}{row['日']:02d}"
    place_str = f"{row['レース場番号']:02d}"
    race_str = f"{row['レース番号']:02d}"
    return int(f"{date_str}{place_str}{race_str}")

In [ ]:
programs_df.insert(0, "レースID", programs_df.apply(make_race_id, axis=1))

In [ ]:
# 級別を数値に変換
def grade_to_numeric(grade):
    if grade == "A1":
        return 3
    elif grade == "A2":
        return 2
    elif grade == "B1":
        return 1
    elif grade == "B2":
        return 0
    else:
        return np.nan


programs_df["級別"] = programs_df["級別"].apply(grade_to_numeric)

In [ ]:
columns_to_drop = [
    "年",
    "月",
    "日",
    "レース場番号",
    "レース番号",
    "距離",
    "投票締切時間",
    "支部",
    "モーター番号",
    "ボート番号",
    "開催日",
]
programs_df.drop(columns=columns_to_drop, inplace=True)

In [ ]:
print("\n=== レースIDの先頭20行 ===")
print(programs_df.head(20))


=== レースIDの先頭20行 ===
           レースID  選手登番  年齢  体重  級別  全国勝率  全国2連率  当地勝率  当地2連率  モーター2連率  ボート2連率
0   202501012401  3527  53  52   1  5.43  32.65  5.97  40.91    29.20   36.21
1   202501012401  4603  36  53   1  4.98  31.40  5.32  33.33    39.20   39.32
2   202501012401  3843  48  54   1  4.74  28.16  5.08  29.09    28.35   36.13
3   202501012401  5048  30  54   1  4.32  24.19  4.60  24.19    24.55   35.29
4   202501012401  5335  20  48   1  2.38   6.67  2.48  10.87    30.58   37.70
5   202501012401  3906  47  60   1  4.03  24.04  4.87  31.47    35.00   32.77
6   202501012402  4004  46  52   1  4.57  21.88  4.86  26.90    43.59   31.93
7   202501012402  4315  40  52   2  5.21  35.90  6.01  44.76    41.53   30.25
8   202501012402  5129  27  47   2  5.86  43.18  5.82  39.34    28.69   35.77
9   202501012402  3722  54  52   1  3.84  17.65  3.83  13.04    30.89   36.67
10  202501012402  4705  36  55   1  4.10  25.32  4.29  25.71    38.26   31.09
11  202501012402  4094  43  54   1  4.00  1

In [ ]:
print("=== データ型情報 ===")
print(programs_df.dtypes)

=== データ型情報 ===
レースID        int64
選手登番         int64
年齢           int64
体重           int64
級別           int64
全国勝率       float64
全国2連率      float64
当地勝率       float64
当地2連率      float64
モーター2連率    float64
ボート2連率     float64
dtype: object


In [ ]:
# results.csvを読み込み
results_df = pd.read_csv("data/results.csv")

In [ ]:
print(f"results.csvの形状: {results_df.shape}")
print("\nカラム一覧:")
print(results_df.columns.tolist())
print("\n先頭5行:")
results_df.head()

results.csvの形状: (185161, 46)

カラム一覧:
['年', '月', '日', 'レース場番号', 'レース番号', '距離', '天候', '風向', '風速', '波高', '単勝_艇番', '単勝_払戻金', '複勝1着_艇番', '複勝1着_払戻金', '複勝2着_艇番', '複勝2着_払戻金', '2連単_艇番', '2連単_払戻金', '2連単_人気', '2連複_艇番', '2連複_払戻金', '2連複_人気', '拡連複1_艇番', '拡連複1_払戻金', '拡連複1_人気', '拡連複2_艇番', '拡連複2_払戻金', '拡連複2_人気', '拡連複3_艇番', '拡連複3_払戻金', '拡連複3_人気', '3連単_艇番', '3連単_払戻金', '3連単_人気', '3連複_艇番', '3連複_払戻金', '3連複_人気', '着順', '選手登番', '艇番', 'モーター番号', 'ボート番号', '展示', '進入', 'スタートタイミング', 'レースタイム']

先頭5行:


,年,月,日,レース場番号,レース番号,距離,天候,風向,風速,波高,...,3連複_人気,着順,選手登番,艇番,モーター番号,ボート番号,展示,進入,スタートタイミング,レースタイム
0,2025,1,1,24,1,1800,晴,北西,2,1,...,2.0,1,3527,1,57,59,6.87,1.0,0.13,1.48.5
1,2025,1,1,24,1,1800,晴,北西,2,1,...,2.0,2,4603,2,29,53,6.82,2.0,0.10,1.49.1
2,2025,1,1,24,1,1800,晴,北西,2,1,...,2.0,4,3843,3,43,48,6.90,3.0,0.14,1.52.0
3,2025,1,1,24,1,1800,晴,北西,2,1,...,2.0,3,5048,4,40,51,6.86,5.0,0.16,1.50.7
4,2025,1,1,24,1,1800,晴,北西,2,1,...,2.0,6,5335,5,65,65,6.91,6.0,0.27,NaN


In [ ]:
results_df.insert(0, "レースID", results_df.apply(make_race_id, axis=1))

In [ ]:
print("\n=== レースIDの先頭20行 ===")
print(results_df.head(20))


=== レースIDの先頭20行 ===
           レースID     年  月  日  レース場番号  レース番号    距離 天候  風向  風速  ...  3連複_人気  着順  \
0   202501012401  2025  1  1      24      1  1800  晴  北西   2  ...     2.0   1   
1   202501012401  2025  1  1      24      1  1800  晴  北西   2  ...     2.0   2   
2   202501012401  2025  1  1      24      1  1800  晴  北西   2  ...     2.0   4   
3   202501012401  2025  1  1      24      1  1800  晴  北西   2  ...     2.0   3   
4   202501012401  2025  1  1      24      1  1800  晴  北西   2  ...     2.0   6   
5   202501012401  2025  1  1      24      1  1800  晴  北西   2  ...     2.0   5   
6   202501012402  2025  1  1      24      2  1800  晴  北西   2  ...     1.0   2   
7   202501012402  2025  1  1      24      2  1800  晴  北西   2  ...     1.0   1   
8   202501012402  2025  1  1      24      2  1800  晴  北西   2  ...     1.0   3   
9   202501012402  2025  1  1      24      2  1800  晴  北西   2  ...     1.0   6   
10  202501012402  2025  1  1      24      2  1800  晴  北西   2  ...     1.0   4   
11  202

In [ ]:
# programs_dfとresults_dfをマージして着順情報を追加
# レースIDと選手登番でマージ
merged_df = programs_df.merge(
    results_df[["レースID", "選手登番", "着順"]],
    on=["レースID", "選手登番"],
    how="left",
)

print(f"マージ前のprograms_df形状: {programs_df.shape}")
print(f"マージ後のmerged_df形状: {merged_df.shape}")

# 着順が取得できたかどうか確認
print(f"\n着順が取得できた件数: {merged_df['着順'].notna().sum()}")
print(f"着順が取得できなかった件数: {merged_df['着順'].isna().sum()}")

# programs_dfを更新
programs_df = merged_df.copy()

print(f"\n最終的なprograms_df形状: {programs_df.shape}")
print("\n着順が追加されたデータの先頭5行:")
programs_df.head()

マージ前のprograms_df形状: (187488, 11)
マージ後のmerged_df形状: (187488, 12)

着順が取得できた件数: 184081
着順が取得できなかった件数: 3407

最終的なprograms_df形状: (187488, 12)

着順が追加されたデータの先頭5行:


,レースID,選手登番,年齢,体重,級別,全国勝率,全国2連率,当地勝率,当地2連率,モーター2連率,ボート2連率,着順
0,202501012401,3527,53,52,1,5.43,32.65,5.97,40.91,29.20,36.21,1
1,202501012401,4603,36,53,1,4.98,31.40,5.32,33.33,39.20,39.32,2
2,202501012401,3843,48,54,1,4.74,28.16,5.08,29.09,28.35,36.13,4
3,202501012401,5048,30,54,1,4.32,24.19,4.60,24.19,24.55,35.29,3
4,202501012401,5335,20,48,1,2.38,6.67,2.48,10.87,30.58,37.70,6


In [ ]:
# 最後に選手登番を削除
programs_df.drop(columns=["選手登番"], inplace=True)

In [ ]:
# train.csvとして保存
programs_df.to_csv("data/train.csv", index=False, encoding="utf-8-sig")
print("\n=== train.csvとして保存完了 ===")
print(f"保存先: data/train.csv")


=== train.csvとして保存完了 ===
保存先: data/train.csv
